In [ ]:
# ============================================================
# Align TinyLlama-1.1B-Chat-v1.0 with semantic-MARS DeBERTa-v3-base RMs


# ============================================================
# Imports
# ============================================================

import os
import gc
import shutil
import random
import numpy as np
import torch

from getpass import getpass
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset
from huggingface_hub import login, HfApi, model_info
from huggingface_hub.utils import RepositoryNotFoundError

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig

from trl import (
    PPOConfig,
    PPOTrainer,
    AutoModelForCausalLMWithValueHead,
    create_reference_model,
)

from trl.core import LengthSampler


# ============================================================
# User config
# ============================================================

HF_TOKEN = getpass("Enter your Hugging Face token: ").strip()
HF_USERNAME = ""

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)

# If one dataset is already done, add it here.
# Example:
# SKIP_DATASETS = {"HHRLHF"}
SKIP_DATASETS = set()


# ============================================================
# Dataset configs: semantic-MARS DeBERTa-v3-base RMs
# ============================================================

DATASET_CONFIGS = {
    "HHRLHF": {
        "dataset_name": "Anthropic/hh-rlhf",
        "prompt_extractor": "hh",

        # semantic-MARS DeBERTa RM trained on HH-RLHF
        "rm_repo": f"{HF_USERNAME}/",

        # output aligned TinyLlama repo
        "output_repo": f"{HF_USERNAME}/",
    },

    "UltraFeedback_openbmb": {
        "dataset_name": "openbmb/UltraFeedback",
        "prompt_extractor": "ultrafeedback",

        # semantic-MARS DeBERTa RM trained on UltraFeedback
        "rm_repo": f"{HF_USERNAME}/",

        # output aligned TinyLlama repo
        "output_repo": f"{HF_USERNAME}/",
    },
}


# ============================================================
# PPO training config
# Matched to existing TinyLlama DeBERTa WoN/baseline/MARS alignment code
# ============================================================

NUM_TRAIN_SAMPLES = 1000
MAX_PROMPT_TOKENS = 256

MIN_NEW_TOKENS = 32
MAX_NEW_TOKENS = 64

LR = 5e-6
BATCH_SIZE = 16
MINI_BATCH_SIZE = 4
GRAD_ACCUM = 4
PPO_EPOCHS = 2
TOTAL_PPO_STEPS = 250

INIT_KL_COEF = 0.02
TARGET_KL = 6.0
ADAP_KL_CTRL = True

REWARD_CLIP = 5.0

# LoRA config for TinyLlama/LLaMA-style architecture
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

SEED = 42


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# Device setup
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
dtype = torch.bfloat16 if bf16_ok else torch.float16

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("dtype:", dtype)


# ============================================================
# Helper functions
# ============================================================

def clean_text(text: str) -> str:
    """Basic whitespace cleanup."""
    return " ".join(str(text).strip().split())


def extract_prompt_from_hh(text: str) -> str:
    """
    Extract dialogue prefix from HH-RLHF chosen/rejected text.
    Keeps the conversation up to the final Assistant: marker.
    """
    text = str(text).strip()

    if "\n\nAssistant:" in text:
        parts = text.rsplit("\n\nAssistant:", 1)
        return (parts[0] + "\n\nAssistant:").strip()

    if "Assistant:" in text:
        parts = text.rsplit("Assistant:", 1)
        return (parts[0] + "Assistant:").strip()

    return clean_text(text)


def extract_prompt(example: dict, extractor_type: str) -> str:
    """
    Extract prompt depending on dataset format.
    """
    if extractor_type == "hh":
        return extract_prompt_from_hh(example["chosen"])

    if extractor_type == "ultrafeedback":
        if "instruction" in example and example["instruction"] is not None:
            return clean_text(example["instruction"])

        if "prompt" in example and example["prompt"] is not None:
            return clean_text(example["prompt"])

        return ""

    raise ValueError(f"Unknown extractor_type: {extractor_type}")


def build_prompt_dataset(
    tokenizer,
    dataset_name: str,
    extractor_type: str,
    num_samples: int = 1000,
):
    """
    Build prompt-only PPO dataset from the original preference dataset.
    This is matched to the existing TinyLlama DeBERTa alignment pipeline.
    """
    raw = load_dataset(dataset_name, split="train")

    prompts = []
    seen = set()

    for ex in raw:
        prompt = extract_prompt(ex, extractor_type)

        if prompt and prompt not in seen:
            seen.add(prompt)
            prompts.append({"query": prompt})

        if len(prompts) >= num_samples:
            break

    ds = Dataset.from_list(prompts)

    def tok_fn(example):
        out = tokenizer(
            example["query"],
            truncation=True,
            max_length=MAX_PROMPT_TOKENS,
            padding=False,
        )

        example["input_ids"] = out["input_ids"]
        example["attention_mask"] = out["attention_mask"]

        return example

    ds = ds.map(tok_fn)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

    return ds


def collator(data):
    """
    PPOTrainer expects lists of tensors rather than one padded tensor batch.

    Correct form:
      keys come from data[0],
      values are lists over examples.

    This fixes:
      NameError: name 'k' is not defined
    """
    return {k: [d[k] for d in data] for k in data[0]}


@torch.no_grad()
def score_with_reward_model(texts, rm_tokenizer, rm_model, batch_size=16):
    """
    Score prompt-response texts using the DeBERTa reward model.
    """
    scores = []
    rm_model.eval()
    rm_device = next(rm_model.parameters()).device

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]

        enc = rm_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )

        enc = {k: v.to(rm_device) for k, v in enc.items()}

        outputs = rm_model(**enc)
        logits = outputs.logits.squeeze(-1)

        if logits.ndim == 0:
            logits = logits.unsqueeze(0)

        scores.extend(logits.float().cpu().tolist())

    return scores


def normalize_and_clip_rewards(reward_scores, clip_value=5.0):
    """
    Normalize rewards within each PPO batch and clip for stability.
    Same as existing WoN/baseline alignment pipeline.
    """
    rewards = np.array(reward_scores, dtype=np.float32)

    mean = rewards.mean()
    std = rewards.std() + 1e-6

    rewards = (rewards - mean) / std
    rewards = np.clip(rewards, -clip_value, clip_value)

    return [
        torch.tensor(float(r), dtype=torch.float32)
        for r in rewards
    ], float(mean), float(std)


def free_memory():
    """Free CPU/GPU memory between dataset runs."""
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def check_hf_model_repo(repo_id: str, token: str):
    try:
        model_info(repo_id, token=token)
        print(f"Found repo: {repo_id}")
        return True

    except RepositoryNotFoundError:
        print(f"Repo not found or not accessible: {repo_id}")
        return False

    except Exception as e:
        print(f"Could not check repo: {repo_id}")
        print("Error:", repr(e))
        return False


def load_deberta_reward_model(rm_repo: str, token: str):
    """
    Load semantic-MARS DeBERTa reward model.

    This matches the existing DeBERTa alignment pipeline:
    - RM tokenizer loaded from RM repo.
    - RM model loaded as AutoModelForSequenceClassification.
    - RM placed on a single device.
    """

    ok = check_hf_model_repo(rm_repo, token)

    if not ok:
        raise ValueError(
            f"\nReward model repo not found or not accessible:\n{rm_repo}\n\n"
            "Fix DATASET_CONFIGS or make sure your token has access."
        )

    rm_tokenizer = AutoTokenizer.from_pretrained(
        rm_repo,
        use_fast=True,
        token=token,
    )

    if rm_tokenizer.pad_token is None:
        rm_tokenizer.pad_token = (
            rm_tokenizer.eos_token
            if rm_tokenizer.eos_token is not None
            else rm_tokenizer.sep_token
        )

    rm_model = AutoModelForSequenceClassification.from_pretrained(
        rm_repo,
        torch_dtype=dtype if torch.cuda.is_available() else torch.float32,
        token=token,
    )

    rm_model.to(device)
    rm_model.eval()

    return rm_tokenizer, rm_model


# ============================================================
# Load policy tokenizer once
# ============================================================

policy_tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=True,
    token=HF_TOKEN,
)

if policy_tokenizer.pad_token is None:
    policy_tokenizer.pad_token = policy_tokenizer.eos_token

policy_tokenizer.padding_side = "left"
policy_tokenizer.truncation_side = "left"

if policy_tokenizer.pad_token_id is None and policy_tokenizer.eos_token_id is not None:
    policy_tokenizer.pad_token_id = policy_tokenizer.eos_token_id

print("Policy tokenizer pad_token:", policy_tokenizer.pad_token)
print("Policy tokenizer pad_token_id:", policy_tokenizer.pad_token_id)
print("Policy tokenizer eos_token_id:", policy_tokenizer.eos_token_id)


# ============================================================
# Check RM repos before training
# ============================================================

print("\nChecking semantic-MARS DeBERTa RM repos...")

for data_tag, cfg in DATASET_CONFIGS.items():
    if data_tag in SKIP_DATASETS:
        print(f"Skipping repo check for {data_tag}.")
        continue

    ok = check_hf_model_repo(cfg["rm_repo"], HF_TOKEN)

    if not ok:
        raise ValueError(
            f"Missing semantic-MARS DeBERTa RM repo for {data_tag}: {cfg['rm_repo']}"
        )


# ============================================================
# Main training loop
# ============================================================

for data_tag, cfg in DATASET_CONFIGS.items():

    if data_tag in SKIP_DATASETS:
        print("\n" + "=" * 100)
        print(f"Skipping dataset: {data_tag}")
        print("=" * 100)
        continue

    dataset_name = cfg["dataset_name"]
    extractor_type = cfg["prompt_extractor"]
    rm_repo = cfg["rm_repo"]
    output_repo = cfg["output_repo"]

    print("\n" + "#" * 100)
    print(f"Preparing dataset: {data_tag} ({dataset_name})")
    print(f"Semantic-MARS DeBERTa RM: {rm_repo}")
    print(f"Output repo: {output_repo}")
    print("#" * 100)

    # ------------------------------------------------------------
    # Build prompt dataset for PPO
    # ------------------------------------------------------------

    train_dataset = build_prompt_dataset(
        policy_tokenizer,
        dataset_name=dataset_name,
        extractor_type=extractor_type,
        num_samples=NUM_TRAIN_SAMPLES,
    )

    print(f"{data_tag} training prompts:", len(train_dataset))

    local_output_dir = f"./aligned_tinyllama_deberta_{data_tag}_semantic_MARS"

    if os.path.exists(local_output_dir):
        shutil.rmtree(local_output_dir)

    api.create_repo(
        repo_id=output_repo,
        token=HF_TOKEN,
        exist_ok=True,
        private=False,
    )

    # ------------------------------------------------------------
    # Load semantic-MARS DeBERTa reward model
    # ------------------------------------------------------------

    print("\nLoading semantic-MARS DeBERTa reward model:")
    print(rm_repo)

    rm_tokenizer, rm_model = load_deberta_reward_model(
        rm_repo=rm_repo,
        token=HF_TOKEN,
    )

    # ------------------------------------------------------------
    # Load TinyLlama policy with value head + LoRA
    # ------------------------------------------------------------

    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )

    model = AutoModelForCausalLMWithValueHead.from_pretrained(
        BASE_MODEL,
        torch_dtype=dtype if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        peft_config=peft_config,
        token=HF_TOKEN,
    )

    model.pretrained_model.config.use_cache = False

    if hasattr(model.pretrained_model.config, "pad_token_id"):
        model.pretrained_model.config.pad_token_id = policy_tokenizer.pad_token_id

    if hasattr(model.pretrained_model.config, "eos_token_id"):
        model.pretrained_model.config.eos_token_id = policy_tokenizer.eos_token_id

    if hasattr(model.pretrained_model, "generation_config"):
        model.pretrained_model.generation_config.pad_token_id = policy_tokenizer.pad_token_id
        model.pretrained_model.generation_config.eos_token_id = policy_tokenizer.eos_token_id

    # Reference model for KL regularization
    ref_model = create_reference_model(model)

    # ------------------------------------------------------------
    # PPO config
    # ------------------------------------------------------------

    ppo_config = PPOConfig(
        model_name=BASE_MODEL,
        learning_rate=LR,
        batch_size=BATCH_SIZE,
        mini_batch_size=MINI_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        optimize_cuda_cache=True,
        ppo_epochs=PPO_EPOCHS,
        seed=SEED,
        log_with=None,
        init_kl_coef=INIT_KL_COEF,
        target=TARGET_KL,
        adap_kl_ctrl=ADAP_KL_CTRL,
    )

    ppo_trainer = PPOTrainer(
        config=ppo_config,
        model=model,
        ref_model=ref_model,
        tokenizer=policy_tokenizer,
        dataset=train_dataset,
        data_collator=collator,
    )

    output_length_sampler = LengthSampler(
        MIN_NEW_TOKENS,
        MAX_NEW_TOKENS,
    )

    generation_kwargs = {
        "do_sample": True,
        "top_p": 0.9,
        "temperature": 0.7,
        "pad_token_id": policy_tokenizer.pad_token_id,
        "eos_token_id": policy_tokenizer.eos_token_id,
    }

    # ------------------------------------------------------------
    # PPO training
    # ------------------------------------------------------------

    step_count = 0

    while step_count < TOTAL_PPO_STEPS:

        progress = tqdm(
            ppo_trainer.dataloader,
            desc=f"{data_tag}-TinyLlama-semantic-MARS-DeBERTa",
            leave=False,
        )

        for batch in progress:

            if step_count >= TOTAL_PPO_STEPS:
                break

            query_tensors = batch["input_ids"]

            # Generate responses from current policy
            response_tensors = ppo_trainer.generate(
                query_tensors,
                return_prompt=False,
                length_sampler=output_length_sampler,
                **generation_kwargs,
            )

            # Some TRL versions return a tensor instead of list[tensor]
            if isinstance(response_tensors, torch.Tensor):
                response_tensors = [r for r in response_tensors]

            # Decode prompts and responses
            queries = [
                policy_tokenizer.decode(q, skip_special_tokens=True)
                for q in query_tensors
            ]

            responses = [
                policy_tokenizer.decode(r, skip_special_tokens=True)
                for r in response_tensors
            ]

            # Reward model scores full prompt-response text
            full_texts = [
                q.strip() + " " + r.strip()
                for q, r in zip(queries, responses)
            ]

            raw_reward_scores = score_with_reward_model(
                full_texts,
                rm_tokenizer,
                rm_model,
                batch_size=16,
            )

            rewards, reward_mean, reward_std = normalize_and_clip_rewards(
                raw_reward_scores,
                clip_value=REWARD_CLIP,
            )

            # PPO update
            stats = ppo_trainer.step(
                query_tensors,
                response_tensors,
                rewards,
            )

            mean_raw_reward = float(np.mean(raw_reward_scores))

            postfix = {
                "step": step_count + 1,
                "raw_reward": f"{mean_raw_reward:.4f}",
                "norm_mean": f"{reward_mean:.4f}",
                "norm_std": f"{reward_std:.4f}",
            }

            # Optional KL logging, useful for debugging fairness/stability
            if isinstance(stats, dict):
                for key in [
                    "objective/kl",
                    "ppo/policy/approxkl",
                    "ppo/policy/policykl",
                    "kl",
                ]:
                    if key in stats:
                        try:
                            postfix["kl"] = f"{float(stats[key]):.4f}"
                            break
                        except Exception:
                            pass

            progress.set_postfix(postfix)

            step_count += 1

    print(
        f"Finished PPO | Dataset: {data_tag} | semantic-MARS DeBERTa RM | "
        f"Total steps = {step_count}"
    )

    # ------------------------------------------------------------
    # Save and push LoRA adapter + tokenizer
    # ------------------------------------------------------------

    os.makedirs(local_output_dir, exist_ok=True)

    ppo_trainer.model.save_pretrained(local_output_dir)
    policy_tokenizer.save_pretrained(local_output_dir)

    with open(os.path.join(local_output_dir, "README.md"), "w") as f:
        f.write(
            f"# {output_repo}\n\n"
            f"Base model: `{BASE_MODEL}`\n\n"
            f"Alignment dataset: `{dataset_name}`\n\n"
            f"Reward model: `{rm_repo}`\n\n"
            f"Method: PPO alignment with LoRA adapters.\n\n"
            f"Reward model type: semantic-distance-aware MARS DeBERTa-v3-base reward model.\n\n"
            f"Training details:\n"
            f"- NUM_TRAIN_SAMPLES: {NUM_TRAIN_SAMPLES}\n"
            f"- MAX_PROMPT_TOKENS: {MAX_PROMPT_TOKENS}\n"
            f"- MIN_NEW_TOKENS: {MIN_NEW_TOKENS}\n"
            f"- MAX_NEW_TOKENS: {MAX_NEW_TOKENS}\n"
            f"- TOTAL_PPO_STEPS: {TOTAL_PPO_STEPS}\n"
            f"- PPO_EPOCHS: {PPO_EPOCHS}\n"
            f"- LR: {LR}\n"
            f"- Batch size: {BATCH_SIZE}\n"
            f"- Mini-batch size: {MINI_BATCH_SIZE}\n"
            f"- Gradient accumulation: {GRAD_ACCUM}\n"
            f"- INIT_KL_COEF: {INIT_KL_COEF}\n"
            f"- TARGET_KL: {TARGET_KL}\n"
            f"- ADAP_KL_CTRL: {ADAP_KL_CTRL}\n"
            f"- Reward normalization and clipping enabled, clip={REWARD_CLIP}\n"
            f"- LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}\n"
            f"- Generation during PPO: do_sample=True, top_p=0.9, temperature=0.7\n"
        )

    print(f"Pushing aligned TinyLlama model to: {output_repo}")

    policy_tokenizer.push_to_hub(
        repo_id=output_repo,
        token=HF_TOKEN,
        commit_message=(
            f"Upload tokenizer for {data_tag} semantic-MARS "
            f"DeBERTa aligned TinyLlama model"
        ),
    )

    api.upload_folder(
        folder_path=local_output_dir,
        repo_id=output_repo,
        repo_type="model",
        token=HF_TOKEN,
        commit_message=(
            f"Upload PPO-aligned TinyLlama-1.1B model using "
            f"semantic-MARS DeBERTa reward model on {data_tag}"
        ),
    )

    print(f"Done: {output_repo}")

    # ------------------------------------------------------------
    # Cleanup before next dataset
    # ------------------------------------------------------------

    del ppo_trainer
    del model
    del ref_model
    del rm_model
    del rm_tokenizer

    free_memory()


print("\nAll semantic-MARS DeBERTa-aligned TinyLlama-1.1B models trained and pushed successfully.")
print("Repos:")

for data_tag, cfg in DATASET_CONFIGS.items():
    print(f"{data_tag}: {cfg['output_repo']}")